# Where airlines lose time: carrier-attributable delay in US domestic flights

*This analysis uses US DOT (BTS) flight data rather than Singapore data, since CAAS/data.gov.sg only publishes aggregate monthly figures for Changi — no flight-level, cause-coded delay data is publicly available. BTS was chosen as the best available public dataset with the granularity needed for this kind of analysis.*

**Question.** Which airports or carriers show disproportionately high carrier-attributable delay, and does time-of-day or day-of-week make it worse?

**What "carrier-attributable" means.** BTS splits each delay of 15+ minutes into five causes. *Carrier* delay covers causes inside the airline's control: maintenance, crew, cleaning, fuelling, baggage. BTS has no separate "mechanical" field, so this is **not** a measure of mechanical problems. Weather, air-traffic-system (NAS) and security delays sit outside the airline's control. Late-arriving-aircraft delay is knock-on delay from an earlier flight and is kept separate.

## Summary

*To be written after the analysis runs on the real data: one finding, one number, one recommendation. Keep it to 3–4 sentences.*

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from aviation_delays import data, metrics, charts

pd.set_option("display.float_format", "{:,.2f}".format)
DATA_PATH = Path(os.environ.get("AVIATION_DATA", data.PROCESSED_PATH))
flights = data.load_processed(DATA_PATH)
af = data.analysis_frame(flights)

charts.apply_style()
FIG_DIR = ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MIN_CARRIER_FLIGHTS = 50_000   # from 01_data_check
MIN_AIRPORT_FLIGHTS = 20_000
FLAG_INDEX = 1.2               # "disproportionate" = 20%+ above the network rate

net = metrics.network_summary(af)
period = f"{af['date'].min():%b %Y} – {af['date'].max():%b %Y}"

## Context: how big is carrier-attributable delay?

In [ ]:
print(f"{period}: {int(net['flights']):,} completed flights")
print(f"Carrier-attributable delay: {net['carrier_min_per_flight']:.1f} min per flight on average, "
      f"{net['carrier_share']:.0%} of all cause-coded delay minutes")
metrics.cause_breakdown(af).assign(hours=lambda d: d["minutes"] / 60).drop(columns="minutes")

## 1. Carriers

In [ ]:
by_carrier = metrics.group_metrics(af, "carrier", min_flights=MIN_CARRIER_FLIGHTS)
# Is a carrier worse because of the airports it flies from? Compare against
# other carriers at the same origin airports.
by_carrier = by_carrier.join(metrics.mix_adjusted(af, "carrier", "origin")[["adj_index"]])
flagged_carriers = by_carrier.index[(by_carrier["index"] >= FLAG_INDEX) & (by_carrier["adj_index"] >= FLAG_INDEX)].tolist()
print("Flagged carriers (raw and airport-mix-adjusted index both >= %.1f):" % FLAG_INDEX, flagged_carriers or "none")
by_carrier

In [ ]:
fig = charts.ranked_bars(
    by_carrier, "carrier_min_per_flight",
    title="Carrier-attributable delay per flight, by airline",
    subtitle=f"Minutes per completed flight, {period}. Highlighted: ≥{FLAG_INDEX:.1f}× network rate, even after controlling for airports flown",
    xlabel="Carrier-attributable delay (min per flight)",
    highlight=flagged_carriers, reference=net["carrier_min_per_flight"], top_n=None,
)
fig.savefig(FIG_DIR / "1_carriers.png", bbox_inches="tight")

**Finding 1.** *To fill in: which carriers stand out, by how much, and whether it survives the airport-mix control.*

## 2. Airports
Grouped by **origin** airport, where the delay was incurred.

In [ ]:
by_airport = metrics.group_metrics(af, "origin", min_flights=MIN_AIRPORT_FLIGHTS)
by_airport = by_airport.join(metrics.mix_adjusted(af, "origin", "carrier")[["adj_index"]])
flagged_airports = by_airport.index[(by_airport["index"] >= FLAG_INDEX) & (by_airport["adj_index"] >= FLAG_INDEX)].tolist()
print("Flagged airports (raw and carrier-mix-adjusted index both >= %.1f):" % FLAG_INDEX, flagged_airports or "none")
by_airport.head(20)

In [ ]:
fig = charts.ranked_bars(
    by_airport, "carrier_min_per_flight",
    title="Airports with the most carrier-attributable delay per departure",
    subtitle=f"Top {min(15, len(by_airport))} of {len(by_airport)} airports with ≥{MIN_AIRPORT_FLIGHTS:,} departures, {period}. Highlighted: ≥{FLAG_INDEX:.1f}× network, even after controlling for carrier mix",
    xlabel="Carrier-attributable delay (min per departure)",
    highlight=flagged_airports, reference=net["carrier_min_per_flight"], top_n=15,
)
fig.savefig(FIG_DIR / "2_airports.png", bbox_inches="tight")

**Finding 2.** *To fill in.*

## 3. Does time of day or day of week make it worse?
By **scheduled** departure hour (local time), so the hour doesn't shift because the flight itself was late.

In [ ]:
by_hour = metrics.group_metrics(af, "dep_hour").sort_index()
by_dow = metrics.group_metrics(af, "day_of_week").sort_index().rename(index=dict(enumerate(data.DAY_NAMES)))
display(by_hour[["flights", "carrier_min_per_flight", "carrier_delayed_rate", "index"]])
by_dow[["flights", "carrier_min_per_flight", "carrier_delayed_rate", "index"]]

In [ ]:
hour_dow = metrics.grid(af[af["dep_hour"].between(5, 23)], "day_of_week", "dep_hour", min_flights=1_000)
hour_dow.index = [data.DAY_NAMES[i] for i in hour_dow.index]
fig = charts.heatmap(
    hour_dow,
    title="Carrier-attributable delay builds through the day",  # check against the data before keeping this title
    subtitle=f"Minutes per completed flight by scheduled departure hour and weekday, {period}",
    xlabel="Scheduled departure hour (local)", colorbar_label="min per flight",
)
fig.savefig(FIG_DIR / "3_hour_weekday.png", bbox_inches="tight")

Does timing make the flagged carriers *worse*, i.e. does their gap over the network widen at certain times?

In [ ]:
bands = pd.cut(af["dep_hour"].astype(int), [0, 9, 13, 17, 21, 24], right=False,
               labels=["<09:00", "09–13", "13–17", "17–21", "21+"])
by_band = af.assign(band=bands).groupby(["band", "carrier"], observed=True)["carrier_delay"].mean().unstack()
network_band = af.assign(band=bands).groupby("band", observed=True)["carrier_delay"].mean()
gap = by_band[flagged_carriers].div(network_band, axis=0) if flagged_carriers else by_band.div(network_band, axis=0)
gap.round(2)  # >1 = worse than network in that time band

**Finding 3.** *To fill in.*

## Key numbers for the writeup

In [ ]:
lines = [f"Period: {period}; {int(net['flights']):,} completed flights."]
lines.append(f"Network: {net['carrier_min_per_flight']:.1f} carrier-attributable min/flight; "
             f"{net['carrier_share']:.0%} of cause-coded delay minutes.")
for c in flagged_carriers:
    r = by_carrier.loc[c]
    lines.append(f"Carrier {c}: {r['carrier_min_per_flight']:.1f} min/flight ({r['index']:.2f}x network, "
                 f"{r['adj_index']:.2f}x after airport mix); {r['excess_hours']:,.0f} excess delay hours.")
for a in flagged_airports:
    r = by_airport.loc[a]
    lines.append(f"Airport {a}: {r['carrier_min_per_flight']:.1f} min/departure ({r['index']:.2f}x network); "
                 f"{r['excess_hours']:,.0f} excess delay hours.")
peak = by_hour.loc[5:23, "carrier_min_per_flight"]
lines.append(f"Hour of day: {peak.idxmin():02d}:00 lowest ({peak.min():.1f} min), {peak.idxmax():02d}:00 highest ({peak.max():.1f} min).")
lines.append(f"Weekday: {by_dow['carrier_min_per_flight'].idxmin()} lowest, {by_dow['carrier_min_per_flight'].idxmax()} highest.")
print("\n".join(lines))

## Recommendation

*To write from the numbers above, e.g.: "If carrier X brought its carrier-attributable delay in the 17:00–21:00 bank down to the network rate, it would remove ~N delay hours a year. Start with the operational causes behind it (turnaround, crew, maintenance scheduling)." Attach exactly one number.*

## Caveats

- **Carrier delay ≠ mechanical delay.** BTS folds maintenance into the same field as crew, cleaning, fuelling and baggage.
- Cause minutes are only recorded for flights 15+ minutes late, so short delays are invisible to this analysis.
- Late-arriving-aircraft delay is partly the carrier's own knock-on delay but can't be split by cause; it is excluded from "carrier-attributable," which makes the estimate conservative.
- BTS covers US carriers above a revenue threshold (domestic flights only), so the results may not carry over to other markets such as Changi.
- Cause attribution is self-reported by airlines.

## Future work (out of scope for this pass)

- Weather-delay analysis by airport and season
- Route and distance effects
- Predicting carrier-attributable delay
- Interactive dashboard (Streamlit)